# EV Prevalence Data Cleaning Notebook

## Research Question

Do ZIP–state areas with higher median household income have higher EV prevalence?

---

## Datasets Used

This project integrates three primary datasets at the ZIP–state level:

1. **EV Prevalence Dataset (EV adoption)** — *10,823 records*  
   This serves as the master EV adoption dataset for the group analysis. It was constructed by aggregating individual EV registration datasets from 14 U.S. states that publicly provide vehicle registration data. The data were cleaned, standardized, and merged into a unified dataset.

2. **ACS Dataset (Socioeconomic variables)** — *33,772 records (raw, before cleaning)*  
   Socioeconomic indicators were obtained from the American Community Survey (ACS) using the Census API:  
   https://api.census.gov/data/2024/acs/acs5  
   This dataset includes variables such as income, education, poverty rate, home value, rent, unemployment, age, and commuting behavior.

3. **EV Charging Dataset (Infrastructure variables)** — *96,019 records (raw, before cleaning)*  
   Charging infrastructure data were obtained from the National Renewable Energy Laboratory (NREL) Alternative Fuel Stations API:  
   https://developer.nrel.gov/docs/transportation/alt-fuel-stations-v1/all/  
   This dataset provides information on charging station availability, charging type, network provider, pricing availability, and workplace charging.

---

## Unit of Analysis

ZIP-state level (one observation per unique zip_state)

where: zip_state = ZIP + state

Example: 90210_CA

---

## Note on Geographic Key Construction

ZIP codes alone are not always reliable unique identifiers for statistical analysis because they are defined by the USPS for mail delivery rather than strict geographic boundaries.

Some ZIP codes may appear associated with multiple states or create ambiguity during dataset merging.

To ensure data integrity, all datasets are merged using a composite geographic key:

zip_state = ZIP + state

This ensures:

- one unique observation per ZIP-state combination

- correct merging across datasets

- avoidance of duplicate key issues

- preservation of geographic consistency

---

## Response Variable

EV_Prevalence = (total_ev / population) × 1000

where:

- total_ev = number of registered electric vehicles

- population = total ZIP-state population

- EV_Prevalence = EVs per 1,000 residents

---

## Key Idea

EV adoption is influenced by:

- Economic capacity (income)

- Infrastructure availability (charging)

- Demographic factors (education, poverty, commuting patterns, age)

The conceptual framework is:

EV Adoption = f(Economic Capacity, Infrastructure, Demographics)


## Variable Description and Purpose

This project integrates three datasets:

1. EV Prevalence dataset (ZIP-state level EV adoption)

2. ACS dataset (ZIP-state level socioeconomic variables)

3. EV Charging dataset (ZIP-state level infrastructure variables)

The goal is to test:

Hypothesis 1 (H1):
Higher-income ZIP-state areas have higher EV prevalence.

---

# 1. KEY IDENTIFIERS

## ZIP
- Description: 5-digit ZIP code
- Purpose: Geographic component of composite key

---

## state
- Description: U.S. state associated with the ZIP code
- Purpose: Geographic component of composite key

---

## zip_state
- Definition: Composite key formed as:

ZIP + state

Example:

90210_CA

- Purpose: Primary key used to merge all datasets
- Role: Defines unit of analysis

One observation = one unique zip_state.

---

# 2. EV PREVALENCE DATASET

## total_ev
- Description: Total number of electric vehicles registered in the ZIP-state area
- Purpose: Measures EV adoption (raw count)
- Role: Used to compute EV_Prevalence and for optional Poisson modeling

---

## Population
- Description: Total population in the ZIP-state area
- Purpose: Used for normalization
- Role: Used only to compute EV prevalence

---

## EV_Prevalence
Definition:

EV_Prevalence = (total_ev / Population) × 1000

- Description: EVs per 1,000 residents
- Purpose: Standardized EV adoption metric
- Role: Primary response variable

---

# 3. ACS (SOCIOECONOMIC DATASET)

## zcta_name
- Description: ZIP area name
- Purpose: Human-readable geographic label
- Role: Not used in modeling

---

## median_income
- Description: Median household income
- Purpose: Proxy for purchasing power
- Role: Primary explanatory variable for H1

Expected:

Higher income → higher EV adoption

---

## population
- Description: Total population
- Purpose: Used in derived variables

---

## poverty_count
- Description: Number of people below poverty line
- Purpose: Measures economic hardship

---

## poverty_rate
Definition:

poverty_count / population

- Purpose: Normalized poverty measure
- Role: Controls for economic inequality

Expected:

Higher poverty → lower EV adoption

---

## bachelors_count
- Description: Number of people with bachelor’s degree

---

## bachelors_rate
Definition:

bachelors_count / population

- Purpose: Education proxy
- Role: Controls for awareness and income potential

Expected:

Higher education → higher EV adoption

---

## median_home_value
- Description: Median housing value
- Purpose: Wealth indicator
- Role: Controls for long-term economic status

---

## median_gross_rent
- Description: Median rent
- Purpose: Cost-of-living indicator
- Role: Controls for urban-rural differences

---

## households
- Description: Number of households
- Purpose: Demographic structure

---

## labor_force
- Description: Number in labor force
- Purpose: Economic activity indicator

---

## unemployed
- Description: Number unemployed

---

## unemployment_rate
Definition:

unemployed / labor_force

- Purpose: Labor market condition
- Role: Controls for economic stability

Expected:

Higher unemployment → lower EV adoption

---

## median_age
- Description: Median age
- Purpose: Demographic control

---

## commute_total
- Description: Total commuters

---

## commute_car
- Description: Number commuting by car

---

## car_commute_rate
Definition:

commute_car / commute_total

- Purpose: Measures personal vehicle dependence
- Role: Relevant to EV adoption

---

## commute_public_transit
- Description: Number using public transit

---

## transit_commute_rate
Definition:

commute_public_transit / commute_total

- Purpose: Measures alternative transportation use
- Role: Controls for reduced EV demand

---

# 4. EV CHARGING DATASET

## ID
- Description: Unique charging station ID
- Purpose: Identifies individual stations
- Role: Not used after aggregation

---

## EV Level1 EVSE Num
- Description: Number of Level 1 charging ports
- Purpose: Basic charging availability

---

## EV Level2 EVSE Num
- Description: Number of Level 2 charging ports
- Purpose: Standard charging infrastructure
- Role: Key infrastructure variable

---

## EV DC Fast Count
- Description: Number of DC fast charging ports
- Purpose: Fast charging availability
- Role: Reduces range anxiety

Expected:

More fast chargers → higher EV adoption

---

## EV Network
- Description: Charging provider
- Purpose: Market structure indicator

---

## EV Pricing
- Description: Charging cost information
- Purpose: Economic barrier indicator

---

## EV Workplace Charging
- Description: Workplace charging availability
- Purpose: Accessibility indicator

---

# 5. DERIVED INFRASTRUCTURE VARIABLES

## num_ports_total
Definition:

Level1 + Level2 + DC Fast

- Purpose: Total charging capacity

---

## ports_per_capita
Definition:

num_ports_total / population

- Purpose: Normalized infrastructure availability
- Role: Better than raw counts

---

## dc_fast_ratio
Definition:

num_dc_fast_ports / num_ports_total

- Purpose: Quality of charging infrastructure

---

# 6. Categorical Variables

The following categorical variables were constructed from continuous predictors using quantile-based grouping to facilitate exploratory analysis and interpretation of nonlinear relationships:

---

## income_group
- Categories: Low / Medium / High  
- Purpose: Examine differences in EV prevalence across income levels and capture potential nonlinear income effects.

---

## poverty_group
- Categories: Low / Medium / High  
- Purpose: Compare EV adoption patterns across varying levels of poverty.

---

## education_group
- Categories: Low / Medium / High  
- Purpose: Analyze how educational attainment relates to EV prevalence.

---

## age_group
- Categories: Young / Middle / Older  
- Purpose: Explore demographic differences in EV adoption.

---

## housing_value_group
- Categories: Low / Medium / High  
- Purpose: Represent wealth distribution and assess its relationship with EV adoption.

---

# FINAL MODEL FRAMEWORK

The dataset supports modeling:

EV_Prevalence = f(
    median_income,
    education,
    poverty,
    infrastructure,
    demographics
)

Where:

- Income = main driver (Hypothesis 1)

- Infrastructure = enabling factor

- Demographics = control variables

This is a ZIP-state-level analysis using one observation per unique zip_state.

In [1]:
# ============================================================
# Load required libraries
# ============================================================
library(dplyr)
library(readr)
library(stringr)
library(jsonlite)
library(ggplot2)

# Apply Times New Roman to all text and plots

theme_set(
  theme_minimal(base_family = "Times New Roman") +
    theme(
      plot.title = element_text(face = "bold"),
      axis.title = element_text(),
      text = element_text(family = "Times New Roman")
    )
)

options(repr.plot.family = "Times New Roman")

cat("
<style>
body, .rendered_html {
    font-family: 'Times New Roman', Times, serif;
    line-height: 1.6;
    font-size: 14px;
}
</style>
")

# ============================================================
# Helper functions used throughout the notebook
# ============================================================

# standardize zipcode
standardize_zip <- function(x) {
  x <- as.character(x)
  x <- substr(x, 1, 5)
  x <- gsub("[^0-9]", "", x)
  sprintf("%05s", x)
}

# standardize state
standardize_state <- function(x) {
  x <- as.character(x)
  x <- str_trim(x)
  str_to_upper(x)
}

# add state_zip key
add_state_zip_key <- function(df, zip_col = "zip", state_col = "state") {
  df %>%
    mutate(
      zip = standardize_zip(.data[[zip_col]]),
      state = standardize_state(.data[[state_col]]),
      state_zip = paste(state, zip, sep = "_")
    )
}

# check duplication on the state_zip key for each dataset
check_dupes <- function(df, key = "state_zip", label = "dataset") {
  cat(label, "duplicated", key, "keys?:", any(duplicated(df[[key]])), "\n")
  cat(label, "rows:", nrow(df), "\n")
  cat(label, "unique", key, "keys:", dplyr::n_distinct(df[[key]]), "\n\n")
}



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union





<style>
body, .rendered_html {
    font-family: 'Times New Roman', Times, serif;
    line-height: 1.6;
    font-size: 14px;
}
</style>


## EV dataset cleaning

In [2]:
# ============================================================
# Read the EV prevalence dataset
# ============================================================
ev_df <- read_csv("FINAL_ev_prevalence.csv", show_col_types = FALSE)

# ============================================================
# Standardize EV column names to lowercase and snake_case
# ============================================================
names(ev_df) <- names(ev_df) %>%
  str_trim() %>%
  str_replace_all("\\s+", "_") %>%
  str_to_lower()


In [3]:
# ============================================================
# Standardize EV keys and create the composite key
# ============================================================
ev_df <- ev_df %>%
  add_state_zip_key(zip_col = "zip", state_col = "state")

# ============================================================
# Keep only valid key values
# ============================================================
ev_df <- ev_df %>%
  filter(
    nchar(zip) == 5,
    zip != "00000",
    !is.na(state),
    state != ""
  )


In [4]:
# ============================================================
# Keep the core EV fields used in later analysis
# ============================================================
ev_df <- ev_df %>%
  dplyr::select(
    zip,
    state,
    state_zip,
    total_ev,
    population,
    ev_prevalence
  ) %>%
  rename(
    population_ev = population
  ) %>%
  distinct(state_zip, .keep_all = TRUE)


In [5]:
# ============================================================
# Create EV grouping variables for EDA
# ============================================================
ev_df <- ev_df %>%
  mutate(
    ev_prevalence_group = case_when(
      ev_prevalence < quantile(ev_prevalence, 0.25, na.rm = TRUE) ~ "Low",
      ev_prevalence < quantile(ev_prevalence, 0.75, na.rm = TRUE) ~ "Medium",
      TRUE ~ "High"
    ),
    ev_count_group = case_when(
      total_ev < quantile(total_ev, 0.25, na.rm = TRUE) ~ "Low",
      total_ev < quantile(total_ev, 0.75, na.rm = TRUE) ~ "Medium",
      TRUE ~ "High"
    ),
    state = factor(state),
    ev_prevalence_group = factor(ev_prevalence_group, levels = c("Low", "Medium", "High")),
    ev_count_group = factor(ev_count_group, levels = c("Low", "Medium", "High"))
  )

# ============================================================
# Quick EV checks
# ============================================================
head(ev_df)
summary(ev_df)
check_dupes(ev_df, label = "EV")


zip,state,state_zip,total_ev,population_ev,ev_prevalence,ev_prevalence_group,ev_count_group
<chr>,<fct>,<chr>,<dbl>,<dbl>,<dbl>,<fct>,<fct>
03901,ME,ME_03901,19,8175,2.324159,Medium,Medium
03902,ME,ME_03902,20,2665,7.504690,Medium,Medium
03903,ME,ME_03903,44,7146,6.157291,Medium,Medium
03904,ME,ME_03904,62,8389,7.390631,Medium,Medium
03905,ME,ME_03905,25,2084,11.996161,Medium,Medium
03906,ME,ME_03906,29,5368,5.402385,Medium,Medium


     zip                state       state_zip            total_ev      
 Length:10823       PA     :1809   Length:10823       Min.   :    0.0  
 Class :character   CA     :1747   Class :character   1st Qu.:    4.0  
 Mode  :character   NY     :1593   Mode  :character   Median :   25.0  
                    TX     :1461                      Mean   :  263.1  
                    NC     : 674                      3rd Qu.:  167.0  
                    MN     : 586                      Max.   :16514.0  
                    (Other):2953                                       
 population_ev    ev_prevalence      ev_prevalence_group ev_count_group
 Min.   :     3   Min.   :   0.000   Low   :2706         Low   :2506   
 1st Qu.:  1408   1st Qu.:   1.945   Medium:5411         Medium:5608   
 Median :  5735   Median :   5.431   High  :2706         High  :2709   
 Mean   : 14203   Mean   :  15.789                                     
 3rd Qu.: 22014   3rd Qu.:  15.208                              

EV duplicated state_zip keys?: FALSE 
EV rows: 10823 
EV unique state_zip keys: 10823 



## Charging dataset cleaning

### Extract the charging data ZIP

Place `alt_fuel_sations-all.csv.zip` in the same folder as this notebook and run with that folder as the working directory. This cell extracts the CSV into `charging_raw/`; the next cell reads it automatically. Rerunning refreshes the extracted CSV.


In [ ]:
# Extract the charging CSV from the ZIP beside this notebook.
# Run the notebook with its folder as the working directory.
zip_path <- "alt_fuel_sations-all.csv.zip"
extract_dir <- "charging_raw"

if (!file.exists(zip_path)) {
  stop(paste("ZIP file not found:", zip_path,
             "\nCurrent working directory:", getwd()))
}

archive_files <- unzip(zip_path, list = TRUE)$Name
csv_files <- archive_files[grepl("\\.csv$", archive_files, ignore.case = TRUE)]

if (length(csv_files) != 1L) {
  stop("Expected exactly one CSV in the ZIP. Found: ", length(csv_files))
}

# Reject paths outside the extraction folder.
if (grepl("(^[/\\\\]|^[A-Za-z]:|(^|[/\\\\])\\.\\.([/\\\\]|$))", csv_files)) {
  stop("The CSV entry contains an unsafe extraction path.")
}

dir.create(extract_dir, showWarnings = FALSE)
unzip(zip_path, files = csv_files, exdir = extract_dir, overwrite = TRUE)
charging_csv_path <- file.path(extract_dir, csv_files)

if (!file.exists(charging_csv_path)) {
  stop("CSV extraction failed: ", charging_csv_path)
}
cat("Charging CSV ready:", charging_csv_path, "\n")


In [ ]:
# ============================================================
# Read the raw charging stations dataset
# ============================================================
stations_df <- read_csv(
  charging_csv_path,
  show_col_types = FALSE
)


In [7]:
# ============================================================
# Keep only electric charging stations opened on or before 12/31/2024
# Open Date is used so charging infrastructure reflects what existed by the end of 2024
# ============================================================
stations_df <- stations_df %>%
  mutate(
    `Open Date` = as.Date(`Open Date`)
  ) %>%
  filter(
    `Fuel Type Code` == "ELEC",
    !is.na(`Open Date`),
    `Open Date` <= as.Date("2024-12-31")
  )


In [8]:
# ============================================================
# Keep only the charging fields needed for analysis
# Use the State field directly from the charging dataset
# ============================================================
stations_df <- stations_df %>%
  dplyr::select(
    State,
    ZIP,
    ID,
    `Open Date`,
    `EV Level1 EVSE Num`,
    `EV Level2 EVSE Num`,
    `EV DC Fast Count`,
    `EV Network`,
    `EV Pricing`,
    `EV Workplace Charging`
  ) %>%
  rename(
    state = State,
    zip = ZIP
  )


In [9]:
# ============================================================
# Standardize charging ZIP and state fields from the charging file
# Then create the helper composite key
# ============================================================
stations_df <- stations_df %>%
  add_state_zip_key(zip_col = "zip", state_col = "state") %>%
  filter(
    nchar(zip) == 5,
    zip != "00000",
    !is.na(state),
    state != ""
  )


In [10]:
# ============================================================
# Replace missing charger counts with 0 before aggregation
# ============================================================
stations_df <- stations_df %>%
  mutate(
    `EV Level1 EVSE Num` = ifelse(is.na(`EV Level1 EVSE Num`), 0, `EV Level1 EVSE Num`),
    `EV Level2 EVSE Num` = ifelse(is.na(`EV Level2 EVSE Num`), 0, `EV Level2 EVSE Num`),
    `EV DC Fast Count` = ifelse(is.na(`EV DC Fast Count`), 0, `EV DC Fast Count`)
  )


In [11]:
# ============================================================
# Aggregate charging data to one row per state + ZIP
# ============================================================
charging_zip_df <- stations_df %>%
  group_by(state, zip, state_zip) %>%
  summarise(
    num_stations = n(),
    num_level1_ports = sum(`EV Level1 EVSE Num`, na.rm = TRUE),
    num_level2_ports = sum(`EV Level2 EVSE Num`, na.rm = TRUE),
    num_dc_fast_ports = sum(`EV DC Fast Count`, na.rm = TRUE),
    num_ports_total = num_level1_ports + num_level2_ports + num_dc_fast_ports,
    network_count = n_distinct(`EV Network`),
    pricing_nonmissing_count = sum(!is.na(`EV Pricing`) & `EV Pricing` != ""),
    workplace_charging_count = sum(`EV Workplace Charging` %in% TRUE, na.rm = TRUE),
    latest_open_date = max(`Open Date`, na.rm = TRUE),
    .groups = "drop"
  )


In [12]:
# ============================================================
# Create continuous charging features
# ============================================================
charging_zip_df <- charging_zip_df %>%
  mutate(
    dc_fast_ratio = ifelse(num_ports_total == 0, 0, num_dc_fast_ports / num_ports_total),
    level2_ratio = ifelse(num_ports_total == 0, 0, num_level2_ports / num_ports_total)
  )


In [13]:
# ============================================================
# Create categorical charging features for EDA and ANOVA
# ============================================================
charging_zip_df <- charging_zip_df %>%
  mutate(
    has_dc_fast = ifelse(num_dc_fast_ports > 0, "Yes", "No"),
    has_level1 = ifelse(num_level1_ports > 0, "Yes", "No"),
    has_level2 = ifelse(num_level2_ports > 0, "Yes", "No"),
    workplace_charging_available = ifelse(workplace_charging_count > 0, "Yes", "No"),
    multi_network = ifelse(network_count > 1, "Yes", "No"),
    charging_density_group = case_when(
      num_ports_total < quantile(num_ports_total, 0.25, na.rm = TRUE) ~ "Low",
      num_ports_total < quantile(num_ports_total, 0.75, na.rm = TRUE) ~ "Medium",
      TRUE ~ "High"
    ),
    fast_charging_group = case_when(
      dc_fast_ratio == 0 ~ "None",
      dc_fast_ratio < quantile(dc_fast_ratio[dc_fast_ratio > 0], 0.5, na.rm = TRUE) ~ "Low",
      TRUE ~ "High"
    ),
    has_dc_fast = factor(has_dc_fast, levels = c("No", "Yes")),
    has_level1 = factor(has_level1, levels = c("No", "Yes")),
    has_level2 = factor(has_level2, levels = c("No", "Yes")),
    workplace_charging_available = factor(workplace_charging_available, levels = c("No", "Yes")),
    multi_network = factor(multi_network, levels = c("No", "Yes")),
    charging_density_group = factor(charging_density_group, levels = c("Low", "Medium", "High")),
    fast_charging_group = factor(fast_charging_group, levels = c("None", "Low", "High"))
  )

# Save cleaned charging dataset
write_csv(charging_zip_df, "ev_charging_2024.csv")

# ============================================================
# Quick charging checks
# ============================================================
head(charging_zip_df)
summary(charging_zip_df)
dim(charging_zip_df)
check_dupes(charging_zip_df, label = "Charging")




state,zip,state_zip,num_stations,num_level1_ports,num_level2_ports,num_dc_fast_ports,num_ports_total,network_count,pricing_nonmissing_count,⋯,latest_open_date,dc_fast_ratio,level2_ratio,has_dc_fast,has_level1,has_level2,workplace_charging_available,multi_network,charging_density_group,fast_charging_group
<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,⋯,<date>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>
AK,99501,AK_99501,4,0,9,0,9,3,1,⋯,2023-12-13,0.0000000,1.0000000,No,No,Yes,Yes,Yes,Medium,None
AK,99503,AK_99503,2,0,3,0,3,2,1,⋯,2022-05-08,0.0000000,1.0000000,No,No,Yes,No,Yes,Medium,None
AK,99514,AK_99514,1,0,1,0,1,1,0,⋯,2019-08-01,0.0000000,1.0000000,No,No,Yes,No,No,Low,None
AK,99515,AK_99515,2,0,2,1,3,1,0,⋯,2022-11-12,0.3333333,0.6666667,Yes,No,Yes,No,No,Medium,Low
AK,99518,AK_99518,1,0,4,0,4,1,0,⋯,2022-12-01,0.0000000,1.0000000,No,No,Yes,No,No,Medium,None
AK,99567,AK_99567,1,0,0,8,8,1,0,⋯,2022-11-05,1.0000000,0.0000000,Yes,No,No,No,No,Medium,High


    state               zip             state_zip          num_stations    
 Length:11489       Length:11489       Length:11489       Min.   :  1.000  
 Class :character   Class :character   Class :character   1st Qu.:  1.000  
 Mode  :character   Mode  :character   Mode  :character   Median :  3.000  
                                                          Mean   :  5.944  
                                                          3rd Qu.:  6.000  
                                                          Max.   :317.000  
 num_level1_ports   num_level2_ports  num_dc_fast_ports num_ports_total 
 Min.   :  0.0000   Min.   :   0.00   Min.   :  0.000   Min.   :   1.0  
 1st Qu.:  0.0000   1st Qu.:   2.00   1st Qu.:  0.000   1st Qu.:   3.0  
 Median :  0.0000   Median :   5.00   Median :  0.000   Median :   8.0  
 Mean   :  0.2302   Mean   :  13.93   Mean   :  4.442   Mean   :  18.6  
 3rd Qu.:  0.0000   3rd Qu.:  13.00   3rd Qu.:  6.000   3rd Qu.:  20.0  
 Max.   :107.0000   Max.   :11

[1] 11489    21

Charging duplicated state_zip keys?: FALSE 
Charging rows: 11489 
Charging unique state_zip keys: 11489 



## ACS dataset cleaning

In [14]:
# ============================================================
# Build ZIP -> dominant state lookup from the cleaned EV data
# This is used only for ACS because ACS ZCTA data does not
# include state directly as a merge-ready field.
# ============================================================
zip_state_lookup <- ev_df %>%
  mutate(
    zip = as.character(zip),
    state = as.character(state)
  ) %>%
  group_by(zip, state) %>%
  summarise(total_ev_state = sum(total_ev, na.rm = TRUE), .groups = "drop") %>%
  group_by(zip) %>%
  slice_max(total_ev_state, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  dplyr::select(zip, state)


In [15]:
# ============================================================
# Step 1: Define the Census API URL
# This pulls ZIP Code Tabulation Area (ZCTA) level ACS 5-year data
# Variables included:
# B19013_001E = Median household income
# B01003_001E = Total population
# B17001_002E = Poverty count
# B15003_022E = Bachelor's degree count
# B25077_001E = Median home value
# B25064_001E = Median gross rent
# B11001_001E = Number of households
# B23025_003E = Labor force
# B23025_005E = Unemployed
# B08301_001E = Total commuters
# B08301_002E = Car commuters
# B08301_010E = Public transit commuters
# B01002_001E = Median age

# ============================================================
# Define the ACS 2024 5-year API URL and pull raw data
# ============================================================
acs_url <- paste0(
  "https://api.census.gov/data/2024/acs/acs5?",
  "get=NAME,",
  "B19013_001E,",
  "B01003_001E,",
  "B17001_002E,",
  "B15003_022E,",
  "B25077_001E,",
  "B25064_001E,",
  "B11001_001E,",
  "B23025_003E,",
  "B23025_005E,",
  "B08301_001E,",
  "B08301_002E,",
  "B08301_010E,",
  "B01002_001E",
  "&for=zip%20code%20tabulation%20area:*"
)

acs_raw <- fromJSON(acs_url)
acs_df <- as.data.frame(acs_raw[-1, ], stringsAsFactors = FALSE)
colnames(acs_df) <- acs_raw[1, ]

# Save raw ACS data
# ============================================================
write_csv(acs_df, "acs_income_raw_2024.csv")



In [16]:
# ============================================================
# Rename ACS fields into readable variable names
# ============================================================
acs_df <- acs_df %>%
  rename(
    zip = `zip code tabulation area`,
    zcta_name = NAME,
    median_income = B19013_001E,
    population = B01003_001E,
    poverty_count = B17001_002E,
    bachelors_count = B15003_022E,
    median_home_value = B25077_001E,
    median_gross_rent = B25064_001E,
    households = B11001_001E,
    labor_force = B23025_003E,
    unemployed = B23025_005E,
    commute_total = B08301_001E,
    commute_car = B08301_002E,
    commute_public_transit = B08301_010E,
    median_age = B01002_001E
  )


In [17]:
# ============================================================
# Convert ACS analysis variables to numeric and clean invalid values
# ------------------------------------------------------------
# ACS fields were imported as character strings, so numeric-like
# columns must be converted to numeric before cleaning. Negative
# placeholder values (e.g., -666666666), which represent missing
# or suppressed estimates, are then recoded as NA.
# ZIP is kept as character to preserve leading zeros.
# ============================================================

# ============================================================
# Convert ACS numeric fields from character to numeric
# ------------------------------------------------------------
# Many ACS fields are imported as character strings. Convert
# analysis variables to numeric first so invalid negative
# placeholders (e.g., -666666666) can be properly recoded as NA.
# ============================================================

acs_df <- acs_df %>%
  mutate(
    across(
      -c(zcta_name, zip),
      ~ suppressWarnings(as.numeric(.))
    )
  ) %>%
  mutate(
    across(where(is.numeric), ~ ifelse(. < 0, NA, .))
  )
summary(acs_df)

acs_df %>%
  summarise(across(where(is.numeric), ~ sum(. < 0, na.rm = TRUE)))


  zcta_name         median_income      population     poverty_count  
 Length:33772       Min.   :  2499   Min.   :     0   Min.   :    0  
 Class :character   1st Qu.: 57505   1st Qu.:   650   1st Qu.:   59  
 Mode  :character   Median : 72067   Median :  2656   Median :  282  
                    Mean   : 78686   Mean   : 10010   Mean   : 1244  
                    3rd Qu.: 91998   3rd Qu.: 13460   3rd Qu.: 1339  
                    Max.   :250001   Max.   :140157   Max.   :33915  
                    NA's   :3225                                     
 bachelors_count median_home_value median_gross_rent   households   
 Min.   :    0   Min.   :   9999   Min.   :  99      Min.   :    0  
 1st Qu.:   53   1st Qu.: 148475   1st Qu.: 814      1st Qu.:  247  
 Median :  259   Median : 221900   Median :1021      Median : 1032  
 Mean   : 1492   Mean   : 297676   Mean   :1182      Mean   : 3864  
 3rd Qu.: 1656   3rd Qu.: 360200   3rd Qu.:1419      3rd Qu.: 5248  
 Max.   :33150   Max.   :2

median_income,population,poverty_count,bachelors_count,median_home_value,median_gross_rent,households,labor_force,unemployed,commute_total,commute_car,commute_public_transit,median_age
<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
0,0,0,0,0,0,0,0,0,0,0,0,0


In [18]:
# ============================================================
# Standardize ACS ZIP and keep only valid ZIP values
# ============================================================
acs_df <- acs_df %>%
  mutate(
    zip = standardize_zip(zip)
  ) %>%
  filter(
    nchar(zip) == 5,
    zip != "00000"
  )


In [19]:
# ============================================================
# ACS data does not include state identifiers at the ZIP level.
# So, add state to ACS using the EV-based ZIP -> state lookup
# Then create the composite key

# ============================================================
acs_df <- acs_df %>%
  left_join(zip_state_lookup, by = "zip") %>%
  filter(!is.na(state), state != "") %>%
  add_state_zip_key(zip_col = "zip", state_col = "state")


In [20]:
# ============================================================
# Convert ACS numeric fields from character to numeric
# ============================================================
acs_df <- acs_df %>%
  mutate(
    median_income = as.numeric(median_income),
    population = as.numeric(population),
    poverty_count = as.numeric(poverty_count),
    bachelors_count = as.numeric(bachelors_count),
    median_home_value = as.numeric(median_home_value),
    median_gross_rent = as.numeric(median_gross_rent),
    households = as.numeric(households),
    labor_force = as.numeric(labor_force),
    unemployed = as.numeric(unemployed),
    commute_total = as.numeric(commute_total),
    commute_car = as.numeric(commute_car),
    commute_public_transit = as.numeric(commute_public_transit),
    median_age = as.numeric(median_age)
  ) %>%
  distinct(state_zip, .keep_all = TRUE)


In [21]:
# ============================================================
# Create derived ACS rates used in EDA and modeling
# ============================================================
acs_df <- acs_df %>%
  mutate(
    poverty_rate = ifelse(population == 0, NA, poverty_count / population),
    bachelors_rate = ifelse(population == 0, NA, bachelors_count / population),
    unemployment_rate = ifelse(labor_force == 0, NA, unemployed / labor_force),
    car_commute_rate = ifelse(commute_total == 0, NA, commute_car / commute_total),
    transit_commute_rate = ifelse(commute_total == 0, NA, commute_public_transit / commute_total)
  )


In [22]:
# ============================================================
# Create categorical ACS group variables for EDA and ANOVA
# ============================================================
acs_df <- acs_df %>%
  mutate(
    income_group = case_when(
      median_income < quantile(median_income, 0.25, na.rm = TRUE) ~ "Low",
      median_income < quantile(median_income, 0.75, na.rm = TRUE) ~ "Medium",
      TRUE ~ "High"
    ),
    poverty_group = case_when(
      poverty_rate < quantile(poverty_rate, 0.25, na.rm = TRUE) ~ "Low",
      poverty_rate < quantile(poverty_rate, 0.75, na.rm = TRUE) ~ "Medium",
      TRUE ~ "High"
    ),
    education_group = case_when(
      bachelors_rate < quantile(bachelors_rate, 0.25, na.rm = TRUE) ~ "Low",
      bachelors_rate < quantile(bachelors_rate, 0.75, na.rm = TRUE) ~ "Medium",
      TRUE ~ "High"
    ),
    age_group = case_when(
      median_age < 35 ~ "Young",
      median_age < 50 ~ "Middle",
      TRUE ~ "Older"
    ),
    housing_value_group = case_when(
      median_home_value < quantile(median_home_value, 0.25, na.rm = TRUE) ~ "Low",
      median_home_value < quantile(median_home_value, 0.75, na.rm = TRUE) ~ "Medium",
      TRUE ~ "High"
    ),
    income_group = factor(income_group, levels = c("Low", "Medium", "High")),
    poverty_group = factor(poverty_group, levels = c("Low", "Medium", "High")),
    education_group = factor(education_group, levels = c("Low", "Medium", "High")),
    age_group = factor(age_group, levels = c("Young", "Middle", "Older")),
    housing_value_group = factor(housing_value_group, levels = c("Low", "Medium", "High"))
  )


In [23]:
# ============================================================
# Save and inspect the cleaned ACS dataset
# ============================================================
write_csv(acs_df, "acs_income_2024.csv")

head(acs_df)
summary(acs_df)
dim(acs_df)
check_dupes(acs_df, label = "ACS")


,zcta_name,median_income,population,poverty_count,bachelors_count,median_home_value,median_gross_rent,households,labor_force,unemployed,⋯,poverty_rate,bachelors_rate,unemployment_rate,car_commute_rate,transit_commute_rate,income_group,poverty_group,education_group,age_group,housing_value_group
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>,<fct>
1,ZCTA5 03901,94771,8175,339,1415,369100,1452,3367,4488,112,⋯,0.04146789,0.1730887,0.02495544,0.7881202,0.0054619936,Medium,Low,Medium,Middle,Medium
2,ZCTA5 03902,133676,2665,126,654,592200,1708,1042,1606,81,⋯,0.04727955,0.2454034,0.05043587,0.8001383,0.0000000000,High,Low,High,Middle,High
3,ZCTA5 03903,101522,7146,343,1400,498900,1948,3040,4146,155,⋯,0.04799888,0.1959138,0.03738543,0.8034456,0.0023492561,Medium,Low,High,Older,Medium
4,ZCTA5 03904,85084,8389,574,1938,448900,1651,3955,4209,209,⋯,0.06842293,0.2310168,0.04965550,0.6912594,0.0002349624,Medium,Medium,High,Middle,Medium
5,ZCTA5 03905,140730,2084,73,362,541100,2533,956,1216,85,⋯,0.03502879,0.1737044,0.06990132,0.5873154,0.0000000000,High,Low,Medium,Older,High
6,ZCTA5 03906,105156,5368,224,778,432900,1105,2065,2765,100,⋯,0.04172876,0.1449329,0.03616637,0.7991535,0.0069257407,High,Low,Medium,Middle,Medium


  zcta_name         median_income      population     poverty_count  
 Length:10823       Min.   :  2499   Min.   :     3   Min.   :    0  
 Class :character   1st Qu.: 63396   1st Qu.:  1408   1st Qu.:  125  
 Mode  :character   Median : 79792   Median :  5735   Median :  521  
                    Mean   : 88724   Mean   : 14203   Mean   : 1693  
                    3rd Qu.:104558   3rd Qu.: 22014   3rd Qu.: 2064  
                    Max.   :250001   Max.   :140157   Max.   :33915  
                    NA's   :553                                      
 bachelors_count median_home_value median_gross_rent   households   
 Min.   :    0   Min.   :   9999   Min.   :  99      Min.   :    0  
 1st Qu.:  150   1st Qu.: 198200   1st Qu.: 950      1st Qu.:  557  
 Median :  692   Median : 311400   Median :1261      Median : 2221  
 Mean   : 2237   Mean   : 410699   Mean   :1413      Mean   : 5349  
 3rd Qu.: 3196   3rd Qu.: 499600   3rd Qu.:1756      3rd Qu.: 8515  
 Max.   :33150   Max.   :2

[1] 10823    27

ACS duplicated state_zip keys?: FALSE 
ACS rows: 10823 
ACS unique state_zip keys: 10823 



## Duplicate-Key Checks and Final Merge

Before merging, each dataset is verified to contain a unique observation per ZIP–state combination to ensure a valid one-to-one merge.

The final modeling dataset is constructed in two steps:

1. **Inner Join (EV + ACS)**  
   The EV and ACS datasets are merged using an `inner_join()` on `(state, zip)`.  
   This ensures that the final sample includes only ZIP–state observations with both:
   - EV adoption data (response variable), and  
   - socioeconomic variables (predictors).

2. **Left Join (Charging Data)**  
   Charging infrastructure data is then merged using a `left_join()` on `(state, zip)`.  
   This preserves ZIP–state observations without charging stations, which are meaningful for analysis.

   Missing charging values are interpreted as **zero infrastructure availability**.

---

### Note on State Assignment for ACS Data

The ACS dataset does not include explicit state identifiers at the ZIP (ZCTA) level.  
To ensure consistent geographic alignment, state information is assigned to ACS records using the EV dataset, which serves as the reference for ZIP–state mapping.

This approach ensures that all datasets are aligned at the ZIP–state level prior to merging.

---

### Summary

- Merge key: `(state, zip)`  
- EV + ACS: `inner_join()` (complete cases for modeling)  
- Charging: `left_join()` (retain zero-infrastructure areas)  

This design balances **data completeness**, **geographic accuracy**, and **model interpretability**.

In [24]:
# ============================================================
# Final duplicate-key checks before merging
# ============================================================
check_dupes(ev_df, label = "EV")
check_dupes(acs_df, label = "ACS")
check_dupes(charging_zip_df, label = "Charging")

# ============================================================
# Optional diagnostics: how many EV rows are not matched?
# ============================================================
cat(
  "EV rows not matched in ACS:",
  nrow(anti_join(ev_df, acs_df, by = c("state", "zip", "state_zip"))),
  "\n"
)
cat(
  "EV rows not matched in Charging:",
  nrow(anti_join(ev_df, charging_zip_df, by = c("state", "zip", "state_zip"))),
  "\n\n"
)

# ============================================================
# Build final modeling dataset
# Step 1: keep only EV rows with ACS matches
# Step 2: preserve rows with no charging stations using left_join
# ============================================================
master_df <- ev_df %>%
  inner_join(acs_df, by = c("state", "zip", "state_zip")) %>%
  left_join(charging_zip_df, by = c("state", "zip", "state_zip"))

# ============================================================
# Replace missing charging values with 0
# Missing here means no matched charging station record for that state + ZIP
# ============================================================

master_df <- master_df %>%
  mutate(
    num_stations = coalesce(num_stations, 0),
    num_level1_ports = coalesce(num_level1_ports, 0),
    num_level2_ports = coalesce(num_level2_ports, 0),
    num_dc_fast_ports = coalesce(num_dc_fast_ports, 0),
    num_ports_total = coalesce(num_ports_total, 0),
    network_count = coalesce(network_count, 0),
    pricing_nonmissing_count = coalesce(pricing_nonmissing_count, 0),
    workplace_charging_count = coalesce(workplace_charging_count, 0),
    dc_fast_ratio = coalesce(dc_fast_ratio, 0),
    level2_ratio = coalesce(level2_ratio, 0),

    has_dc_fast = factor(
      ifelse(is.na(has_dc_fast), "No", as.character(has_dc_fast)),
      levels = levels(charging_zip_df$has_dc_fast)
    ),
    has_level1 = factor(
      ifelse(is.na(has_level1), "No", as.character(has_level1)),
      levels = levels(charging_zip_df$has_level1)
    ),
    has_level2 = factor(
      ifelse(is.na(has_level2), "No", as.character(has_level2)),
      levels = levels(charging_zip_df$has_level2)
    ),
    workplace_charging_available = factor(
      ifelse(is.na(workplace_charging_available), "No", as.character(workplace_charging_available)),
      levels = levels(charging_zip_df$workplace_charging_available)
    ),
    multi_network = factor(
      ifelse(is.na(multi_network), "No", as.character(multi_network)),
      levels = levels(charging_zip_df$multi_network)
    ),
    charging_density_group = factor(
      ifelse(is.na(charging_density_group), "Low", as.character(charging_density_group)),
      levels = levels(charging_zip_df$charging_density_group)
    ),
    fast_charging_group = factor(
      ifelse(is.na(fast_charging_group), "None", as.character(fast_charging_group)),
      levels = levels(charging_zip_df$fast_charging_group)
    )
  )

# ============================================================
# Final merged-dataset checks
# ============================================================
cat("Final merged rows:", nrow(master_df), "\n")
cat("Unique final state_zip keys:", dplyr::n_distinct(master_df$state_zip), "\n")
cat("Any duplicated final state_zip keys?:", any(duplicated(master_df$state_zip)), "\n")

head(master_df)
summary(master_df)


EV duplicated state_zip keys?: FALSE 
EV rows: 10823 
EV unique state_zip keys: 10823 

ACS duplicated state_zip keys?: FALSE 
ACS rows: 10823 
ACS unique state_zip keys: 10823 

Charging duplicated state_zip keys?: FALSE 
Charging rows: 11489 
Charging unique state_zip keys: 11489 

EV rows not matched in ACS: 0 
EV rows not matched in Charging: 5590 

Final merged rows: 10823 
Unique final state_zip keys: 10823 
Any duplicated final state_zip keys?: FALSE 


zip,state,state_zip,total_ev,population_ev,ev_prevalence,ev_prevalence_group,ev_count_group,zcta_name,median_income,⋯,latest_open_date,dc_fast_ratio,level2_ratio,has_dc_fast,has_level1,has_level2,workplace_charging_available,multi_network,charging_density_group,fast_charging_group
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<chr>,<dbl>,⋯,<date>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>
03901,ME,ME_03901,19,8175,2.324159,Medium,Medium,ZCTA5 03901,94771,⋯,NA,0.0000000,0.0000000,No,No,No,No,No,Low,None
03902,ME,ME_03902,20,2665,7.504690,Medium,Medium,ZCTA5 03902,133676,⋯,2016-10-13,0.0000000,1.0000000,No,No,Yes,No,No,Medium,None
03903,ME,ME_03903,44,7146,6.157291,Medium,Medium,ZCTA5 03903,101522,⋯,NA,0.0000000,0.0000000,No,No,No,No,No,Low,None
03904,ME,ME_03904,62,8389,7.390631,Medium,Medium,ZCTA5 03904,85084,⋯,2024-07-19,0.1923077,0.4615385,Yes,Yes,Yes,Yes,Yes,High,Low
03905,ME,ME_03905,25,2084,11.996161,Medium,Medium,ZCTA5 03905,140730,⋯,NA,0.0000000,0.0000000,No,No,No,No,No,Low,None
03906,ME,ME_03906,29,5368,5.402385,Medium,Medium,ZCTA5 03906,105156,⋯,2022-09-23,0.0000000,1.0000000,No,No,Yes,No,No,Low,None


     zip               state            state_zip            total_ev      
 Length:10823       Length:10823       Length:10823       Min.   :    0.0  
 Class :character   Class :character   Class :character   1st Qu.:    4.0  
 Mode  :character   Mode  :character   Mode  :character   Median :   25.0  
                                                          Mean   :  263.1  
                                                          3rd Qu.:  167.0  
                                                          Max.   :16514.0  
                                                                           
 population_ev    ev_prevalence      ev_prevalence_group ev_count_group
 Min.   :     3   Min.   :   0.000   Low   :2706         Low   :2506   
 1st Qu.:  1408   1st Qu.:   1.945   Medium:5411         Medium:5608   
 Median :  5735   Median :   5.431   High  :2706         High  :2709   
 Mean   : 14203   Mean   :  15.789                                     
 3rd Qu.: 22014   3rd Qu.:  15.2

## Final Cleaned Dataset

The final merged dataset is prepared for exploratory data analysis and regression modeling. Datasets are aligned using the composite geographic key `(state, zip)` with `state_zip` retained as a helper identifier, ensuring consistent ZIP–state level observations across all sources.

The EV and ACS datasets are merged using an inner join to retain observations with both EV adoption data and socioeconomic information. Charging infrastructure data are then merged using a left join so that ZIP–state areas without observed charging stations remain in the dataset.

Approximately 52% of ZIP–state observations do not have matching charging infrastructure records. These cases are not data errors; rather, they represent areas with no observed charging stations. Missing infrastructure values are therefore interpreted as **zero infrastructure availability**.

Retaining these observations is essential because it allows the analysis to compare areas with and without charging infrastructure, which is central to the research question.


In [25]:
# ============================================================
# Save final merged dataset
# ============================================================
write_csv(master_df, "ev_income_master.csv")


### Final Dataset Validation

This section reloads the exported master dataset and performs key validation checks to ensure data integrity, confirm unique keys, and verify proper aggregation for downstream analysis.

In [26]:

# ------------------------------------------------------------
# Load the final cleaned dataset
# ------------------------------------------------------------

# read_csv(): reads the saved CSV file into R
master_check <- read_csv("ev_income_master.csv", show_col_types = FALSE)

# ------------------------------------------------------------
# Check for duplicate keys (state_zip should be unique)
# ------------------------------------------------------------
stopifnot(!any(duplicated(master_check$state_zip)))

# ------------------------------------------------------------
# Confirm one row per unique state_zip
# ------------------------------------------------------------

stopifnot(nrow(master_check) == n_distinct(master_check$state_zip))

# ------------------------------------------------------------
# Print row count vs unique key count (sanity check)
# ------------------------------------------------------------

cat("Rows / Unique state_zip: ", nrow(master_check), " / ", n_distinct(master_check$state_zip), "\n", sep = "")

Rows / Unique state_zip: 10823 / 10823


### Data Cleaning Summary

The raw datasets were cleaned, standardized, and merged into a single analytical dataset. Key preprocessing steps included standardizing ZIP and state identifiers, constructing the ZIP–state key, aggregating charging infrastructure to the ZIP–state level, creating socioeconomic and infrastructure-derived variables, and preserving zero-infrastructure areas. 
The final dataset provides a consistent foundation for the analysis notebook.


### Appendix: Files Generated During Data Cleaning

The following files document the progression from raw data to the final analytical dataset:

- **ev_income_master.csv**  
  Final cleaned and integrated dataset used for exploratory data analysis (EDA) and modeling.

- **FINAL_ev_prevalence.csv**  
  Primary EV prevalence dataset serving as the base dataset for all merges.

- **acs_income_2024.csv**  
  Processed ACS dataset after cleaning, derived-variable creation, and alignment with the EV prevalence data.

- **ev_charging_2024.csv**  
  Processed EV charging dataset after cleaning, aggregation, and alignment with the EV prevalence data.

- **acs_income_raw_2024.csv**  
  Raw ACS dataset obtained from the Census API.

- **alt_fuel_stations_all.csv**  
  Raw EV charging dataset obtained from the NREL Alternative Fuel Stations API.
